# 02 — Ingesta y validación del universo

**Capa 2 — Entradas**

**Objetivo:** descargar los precios diarios del universo, verificar su calidad y dejar listo un dataset validado para el procesamiento. Una serie que no pasa las validaciones **no llega al reporte** y queda registrada en el log.

**Entradas:** Yahoo Finance vía `yfinance` (pública, sin credenciales).
**Salidas:**
- `data/raw/precios_diarios.csv`: descarga original con metadatos.
- `data/processed/precios_validados.csv`: solo series válidas.
- `data/processed/resumen_validacion.csv`: resultado de cada validación por ticker.
- KPIs IL3: % de series validadas (meta ≥ 98%) y % de columnas documentadas (meta 100%).

La lógica vive en `src/ingesta.py`; los parámetros, en `src/config.py`.

In [1]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import config as C

# Los mensajes del log (validaciones, exclusiones, alertas) se muestran en el notebook
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(name)s | %(message)s",
                    stream=sys.stdout, force=True)
pd.set_option("display.width", 200)

from src import ingesta as I

## 1. Universo

Las 9 acciones de IA del universo más dos índices:
- **^IXIC (Nasdaq Composite):** referencia sectorial y base de la validación histórica (incluye la puntocom).
- **^GSPC (S&P 500):** mercado para calcular el beta (CAPM).

In [2]:
universo = pd.DataFrame(
    [(t, n, "acción") for t, n in C.UNIVERSO.items()] + [(t, n, "índice") for t, n in C.INDICES.items()],
    columns=["ticker", "nombre", "tipo"])
TICKERS = universo["ticker"].tolist()
universo

,ticker,nombre,tipo
0,NVDA,NVIDIA,acción
1,MSFT,Microsoft,acción
2,GOOGL,Alphabet,acción
3,META,Meta Platforms,acción
4,AMD,Advanced Micro Devices,acción
5,AVGO,Broadcom,acción
6,TSM,Taiwan Semiconductor,acción
7,ORCL,Oracle,acción
8,AMZN,Amazon,acción
9,^IXIC,Nasdaq Composite,índice


## 2. Descarga

Se descargan precios diarios desde el 1-ene-1995 (o desde la salida a bolsa, si es posterior) hasta el último día disponible. A cada fila se le agregan dos metadatos: **fuente** y **fecha/hora de descarga**, para saber siempre de dónde y cuándo vino el dato (trazabilidad).

`auto_adjust=False` conserva ambas columnas: `close` (ajustado solo por splits) y `adj_close` (ajustado por splits y dividendos). Todos los retornos se calculan con `adj_close`, que refleja lo que realmente ganó un inversionista.

In [3]:
crudo = I.descargar(TICKERS)
crudo.head()

INFO     radar.ingesta | Descarga: 11 de 11 tickers con datos, 75320 filas


Price,ticker,fecha,open,high,low,close,adj_close,volume,fuente,fecha_descarga
0,NVDA,1999-01-22,0.043750,0.048828,0.038802,0.041016,0.037517,2.714688e+09,Yahoo Finance (yfinance),2026-11-01 23:52:35
1,NVDA,1999-01-25,0.044271,0.045833,0.041016,0.045313,0.041448,5.104800e+08,Yahoo Finance (yfinance),2026-11-01 23:52:35
2,NVDA,1999-01-26,0.045833,0.046745,0.041146,0.041797,0.038231,3.432000e+08,Yahoo Finance (yfinance),2026-11-01 23:52:35
3,NVDA,1999-01-27,0.041927,0.042969,0.039583,0.041667,0.038113,2.443680e+08,Yahoo Finance (yfinance),2026-11-01 23:52:35
4,NVDA,1999-01-28,0.041667,0.041927,0.041276,0.041536,0.037993,2.275200e+08,Yahoo Finance (yfinance),2026-11-01 23:52:35


## 3. Validaciones

Se aplican las reglas de validación de la Capa 2 (`docs/metodologia.md`, sección 1), ticker por ticker:

| # | Regla | Si falla |
|---|---|---|
| 1 | Sin fechas duplicadas | Se excluye la serie |
| 2 | Precios (open, high, low, close, adj close) positivos | Se excluye la serie |
| 3 | Volumen positivo (solo acciones) | **Se elimina solo ese día** y se registra (error puntual de la fuente). A los índices no se les exige: no tienen volumen propio |
| 4 | Sin huecos mayores a 5 días hábiles (índices: 10), incluido el final de la serie frente a la última fecha del universo | Se excluye la serie. En índices, cada hueco mayor a 5 días se registra en el log |
| 5 | Variación diaria mayor a 50% | Se **marca para revisión** (columna `revisar`); no excluye |

Además, una fila con valores faltantes excluye la serie (dato incompleto).

**Reglas para índices** (tickers que empiezan con `^`): Yahoo informa volumen 0 para muchos índices durante décadas, y los feriados largos de cada bolsa son cierres reales. Por eso a los índices no se les exige volumen positivo y se les acepta un hueco de hasta 10 días hábiles.

**¿Por qué el hueco se mide en días hábiles?** Fines de semana y feriados no son huecos. El mayor hueco esperable en una serie sana es el cierre por el 11-S (4 días hábiles, del 11 al 14 de septiembre de 2001).

**¿Por qué el hueco también se mide al final?** Si la descarga de un ticker se detiene (por ejemplo, Yahoo deja de actualizarlo), su última fecha queda atrás del resto del universo. Sin este chequeo, el reporte mostraría un semáforo con datos viejos sin avisar.

In [4]:
validados, resumen, kpi = I.validar_universo(crudo, TICKERS)
resumen

INFO     radar.ingesta | NVDA: serie válida (6987 días)


INFO     radar.ingesta | MSFT: serie válida (8011 días)


INFO     radar.ingesta | GOOGL: serie válida (5586 días)


INFO     radar.ingesta | META: serie válida (3634 días)


WARNING  radar.ingesta | AMD: día 2015-01-02 eliminado por volumen no positivo


WARNING  radar.ingesta | AMD: variación diaria de 52% el 2016-04-22, marcada para revisión


... (4 outputs left out)

INFO     radar.ingesta | AMZN: serie válida (7412 días)


INFO     radar.ingesta | ^IXIC: serie válida (8011 días)


INFO     radar.ingesta | ^GSPC: serie válida (8011 días)


,ticker,n_dias,fecha_inicio,fecha_fin,duplicados,filas_incompletas,dias_precio_no_positivo,dias_volumen_cero_eliminados,max_hueco_dias_habiles,saltos_marcados_revision,valida,motivo_exclusion
0,NVDA,6987,1999-01-22,2026-10-30,0,0,0,0,4,0,True,
1,MSFT,8011,1995-01-03,2026-10-30,0,0,0,0,4,0,True,
2,GOOGL,5586,2004-08-19,2026-10-30,0,0,0,0,2,0,True,
3,META,3634,2012-05-18,2026-10-30,0,0,0,0,2,0,True,
4,AMD,8010,1995-01-03,2026-10-30,0,0,0,1,4,1,True,
5,AVGO,4336,2009-08-06,2026-10-30,0,0,0,0,2,0,True,
6,TSM,7310,1997-10-09,2026-10-30,0,0,0,0,4,0,True,
7,ORCL,8011,1995-01-03,2026-10-30,0,0,0,0,4,0,True,
8,AMZN,7412,1997-05-15,2026-10-30,0,0,0,0,4,0,True,
9,^IXIC,8011,1995-01-03,2026-10-30,0,0,0,0,4,0,True,


**Cómo leerlo:** cada fila es un ticker; las columnas numéricas cuentan los problemas encontrados por regla. `valida = True` significa que la serie pasa al procesamiento. Los mensajes `WARNING` de arriba muestran los días eliminados por volumen cero y los saltos marcados para revisión.

## 4. Casos marcados para revisión

In [5]:
marcados = validados[validados["revisar"]].copy()
marcados["variacion_diaria"] = (
    validados.groupby("ticker")["adj_close"].pct_change().loc[marcados.index])
marcados[["ticker", "fecha", "close", "adj_close", "variacion_diaria"]]

Price,ticker,fecha,close,adj_close,variacion_diaria
29581,AMD,2016-04-22,3.99,3.99,0.522901


**Cómo leerlo:** un salto mayor a 50% en un día puede ser un error de datos (un split mal ajustado, por ejemplo) o un evento real. Este se dejó en la serie porque el precio se mantiene en el nuevo nivel en los días siguientes (un error de datos típicamente se revierte al día siguiente). Coincide con la publicación de resultados trimestrales de AMD en abril de 2016; el responsable debe confirmarlo y dejar constancia.

## 5. KPIs IL3 (calidad de datos)

In [6]:
pct_columnas, sin_documentar = I.kpi_columnas_documentadas(validados)

pd.DataFrame([
    {"KPI": "% de series con historia completa y validada", "valor": f"{kpi['pct_validadas']:.1%}",
     "meta": f"≥ {kpi['meta']:.0%}", "cumple": "Sí" if not kpi["alerta"] else "No (ALERTA)"},
    {"KPI": "% de columnas documentadas en el diccionario", "valor": f"{pct_columnas:.1%}",
     "meta": "100%", "cumple": "Sí" if pct_columnas == 1 else f"No: {sin_documentar}"},
]).set_index("KPI")

,valor,meta,cumple
KPI,,,
% de series con historia completa y validada,100.0%,≥ 98%,Sí
% de columnas documentadas en el diccionario,100.0%,100%,Sí


**Nota sobre la meta de 98%:** con 11 series, una sola exclusión deja el KPI en 90,9%, bajo la meta. En la práctica, la meta exige que **todas** las series pasen, y cualquier falla dispara la alerta. Es el comportamiento deseado para un universo chico: el comité no debería recibir un reporte incompleto sin aviso.

## 6. Diccionario de datos

El diccionario (`docs/diccionario_datos.csv`) define cada columna del dataset con su unidad y su origen. El KPI anterior verifica automáticamente que ninguna columna quede sin documentar: si en el futuro se agrega una columna y no se documenta, el KPI baja de 100%.

In [7]:
pd.read_csv(C.DIR_DOCS / "diccionario_datos.csv")

,columna,definicion,unidad,tipo,origen
0,ticker,Símbolo bursátil de la acción o índice en Yaho...,texto,str,Yahoo Finance
1,fecha,Día de negociación (hora de Nueva York),fecha AAAA-MM-DD,date,Yahoo Finance
2,open,Precio de apertura del día (ajustado por splits),USD por acción (puntos para índices),float,Yahoo Finance
3,high,Precio máximo del día (ajustado por splits),USD por acción (puntos para índices),float,Yahoo Finance
4,low,Precio mínimo del día (ajustado por splits),USD por acción (puntos para índices),float,Yahoo Finance
5,close,Precio de cierre del día (ajustado por splits),USD por acción (puntos para índices),float,Yahoo Finance
6,adj_close,Precio de cierre ajustado por splits y dividen...,USD por acción (puntos para índices),float,Yahoo Finance
7,volume,Número de acciones transadas en el día,acciones (unidades),int,Yahoo Finance
8,revisar,Verdadero si la variación diaria de adj_close ...,booleano,bool,Calculado (src/ingesta.py)
9,fuente,Origen de los datos,texto,str,Calculado (src/ingesta.py)


## 7. Guardado

Se guardan la descarga original (`data/raw`) y los datos validados más el resumen (`data/processed`), en CSV para que cualquiera pueda reproducir el análisis sin volver a descargar.

In [8]:
I.guardar(crudo, validados, resumen)
print(f"Filas validadas: {len(validados):,} | Tickers: {validados['ticker'].nunique()} | "
      f"Período: {validados['fecha'].min():%d-%m-%Y} a {validados['fecha'].max():%d-%m-%Y}")

INFO     radar.ingesta | Archivos guardados en data/raw y data/processed


Filas validadas: 75,319 | Tickers: 11 | Período: 03-01-1995 a 30-10-2026


## Conclusión

- Las 11 series pasan las validaciones: **KPI IL3 = 100%** (meta ≥ 98%), sin alerta.
- Se eliminó un día con volumen cero en AMD (02-01-2015), registrado en el log. ^GSPC informa volumen 0 el 24-05-2023, pero como es un índice el volumen no se exige y ese día se conserva.
- Se marcó un salto diario mayor a 50% para revisión (AMD, abril de 2016), que corresponde a un movimiento real.
- **100% de las columnas documentadas** en el diccionario de datos.

El dataset validado es la entrada del notebook 03 (variables y semáforo).